In [2]:
from pathlib import Path
import sys
import pandas as pd

# Make project root importable whether CWD is repo root or notebooks/
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'src').exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.io import load_run, save_labeled_run
from src.labels import load_label_config, discover_labeled_runs, label_run_sensors, validate_label_transitions


In [3]:
# Discover all runs with their label configurations
raw_data_dir = PROJECT_ROOT / "data" / "raw"
runs_with_labels = discover_labeled_runs(raw_data_dir)

print(f"Found {len(runs_with_labels)} run(s) with label config(s):")
for run_dir, config_path in runs_with_labels.items():
    print(f"  - {run_dir.name} -> {config_path.relative_to(PROJECT_ROOT)}")


Found 1 run(s) with label config(s):
  - log_20260216_114652.530 -> data/raw/test_1/log_20260216_114652.530/labels_config.json


In [4]:
# Load all runs with their label configurations
all_runs = {}

for run_dir, config_path in runs_with_labels.items():
    label_configs = load_label_config(config_path)
    run = load_run(run_dir, include_pose=True)
    
    all_runs[run.run_id] = {
        'run': run,
        'label_configs': label_configs,
        'config_path': config_path
    }
    
    print(f"\n{run.run_id}:")
    print(f"  Label config: {config_path.relative_to(PROJECT_ROOT)}")
    print(f"  Labels: {', '.join(label_configs.keys())}")
    print(f"  Samples: ACC={len(run.acc)}, GYRO={len(run.gyro)}, ODO={len(run.odo)}, POSE={len(run.pose) if run.pose is not None else 'N/A'}")



log_20260216_114652.530:
  Label config: data/raw/test_1/log_20260216_114652.530/labels_config.json
  Labels: rough_terrain, smooth_plus_rough, smooth_terrain, collision
  Samples: ACC=145708, GYRO=147427, ODO=219888, POSE=351858


In [5]:
# Apply labels to all sensors for each run
labeled_runs = {}

for run_id, run_data in all_runs.items():
    run = run_data['run']
    label_configs = run_data['label_configs']
    
    # Label all sensors
    labeled_sensors = label_run_sensors(run, label_configs)
    labeled_runs[run_id] = labeled_sensors
    
    print(f"\n{run_id.upper()} - Label Distribution:")
    for sensor_name, labeled_df in labeled_sensors.items():
        label_counts = labeled_df['label'].value_counts(dropna=False)
        print(f"  {sensor_name}: {dict(label_counts)}")



LOG_20260216_114652.530 - Label Distribution:
  acc: {nan: np.int64(128559), 'smooth_terrain': np.int64(6835), 'rough_terrain': np.int64(4564), 'smooth_plus_rough': np.int64(3416), 'collision': np.int64(2334)}
  gyro: {nan: np.int64(130278), 'smooth_terrain': np.int64(6835), 'rough_terrain': np.int64(4564), 'smooth_plus_rough': np.int64(3416), 'collision': np.int64(2334)}
  odo: {nan: np.int64(194174), 'smooth_terrain': np.int64(10250), 'rough_terrain': np.int64(6841), 'smooth_plus_rough': np.int64(5123), 'collision': np.int64(3500)}
  pose: {nan: np.int64(310724), 'smooth_terrain': np.int64(16400), 'rough_terrain': np.int64(10937), 'smooth_plus_rough': np.int64(8197), 'collision': np.int64(5600)}


In [6]:
# Validate label assignments
for run_id, labeled_sensors in labeled_runs.items():
    print(f"\n{'='*60}")
    print(f"Validation: {run_id}")
    print('='*60)
    
    # Validate transitions in accelerometer data
    acc_labeled = labeled_sensors['acc']
    validation = validate_label_transitions(acc_labeled, max_transitions=3)
    
    print(f"Total transitions: {validation['total_transitions']}")
    for transition in validation['transition_samples']:
        print(f"\nTransition {transition['transition_num']} (row {transition['row_idx']}):")
        print(transition['samples'])



Validation: log_20260216_114652.530
Total transitions: 128564

Transition 1 (row 1):
              t label
0  1.771239e+09   NaN
1  1.771239e+09   NaN
2  1.771239e+09   NaN
3  1.771239e+09   NaN

Transition 2 (row 2):
              t label
0  1.771239e+09   NaN
1  1.771239e+09   NaN
2  1.771239e+09   NaN
3  1.771239e+09   NaN
4  1.771239e+09   NaN

Transition 3 (row 3):
              t label
1  1.771239e+09   NaN
2  1.771239e+09   NaN
3  1.771239e+09   NaN
4  1.771239e+09   NaN
5  1.771239e+09   NaN


In [7]:
# Save all labeled runs
from dataclasses import replace

output_root = PROJECT_ROOT / "data" / "labeled"

for run_id, run_data in all_runs.items():
    run = run_data['run']
    labeled_sensors = labeled_runs[run_id]
    
    # Create labeled RunData
    run_labeled = replace(
        run,
        acc=labeled_sensors['acc'],
        gyro=labeled_sensors['gyro'],
        odo=labeled_sensors['odo'],
        pose=labeled_sensors.get('pose')
    )
    
    # Save to disk
    save_labeled_run(run_labeled, output_root)
    
    print(f"\nSaved {run_id} to {output_root / run_id}")
    for f in sorted((output_root / run_id).glob("*.csv")):
        file_lines = len(pd.read_csv(f))
        print(f"  {f.name}: {file_lines} rows")



Saved log_20260216_114652.530 to /Users/pratyush/Desktop/DTU/Bachelor_thesis/BSC_Thesis_intrinsic_sensor_analysis/data/labeled/log_20260216_114652.530
  log_t0_acc_1.csv: 145708 rows


/var/folders/wh/6d7zf7cx3632z8j44tx9fqxr0000gn/T/ipykernel_56377/2302610568.py:24: DtypeWarning: Columns (0: label) have mixed types. Specify dtype option on import or set low_memory=False.
  file_lines = len(pd.read_csv(f))


  log_t0_encoder_velocity.csv: 219888 rows
  log_t0_gyro_1.csv: 147427 rows
  log_t0_pose.csv: 351858 rows
